In [1]:
pip install -qq langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -qq langchain-classic

Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_core.tools import tool

# Classic agent imports
from langchain_classic.agents import (
    create_tool_calling_agent,
    AgentExecutor
)

from langchain_openai import ChatOpenAI

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    PromptTemplate,
    MessagesPlaceholder
)

**1.) langchain_core.tool:**
- Purpose: Create tools that an LLM can call.
- A tool is simply a Python function that you make available to an AI model.

**2.) create_tool_calling_agent:**
- Purpose: Create an agent that can decide which tools to call based on the user's request.
- It connects:
    - Your LLM.
    - Your available tools.
    - Your prompt template.
- It creates the agent. It does not, by itself, execute the complete agent workflow.
- For example:
    User asks: "Multiply 25 by 4."
    The agent can identify that the multiply tool is relevant and prepare a tool call.

**3.) AgentExecutor:**
- Purpose: Execute the agent and manage its tool-calling workflow.
- For example:
    1. Receives the user's question.
    2. Sends the request to the agent.
    3. The agent decides to call multiply.
    4. The executor runs the Python function.
    5. The tool returns 100.
    6. The executor passes the result back to the agent.
    7. The agent produces the final response.
- Think of it this way:
    - create_tool_calling_agent = Creates the decision-making component.
    - AgentExecutor = Runs the decision-making component and its tools.

**4.) from langchain_openai import ChatOpenAI:**
- Purpose: Connect LangChain applications to OpenAI chat models.
- It is a integration package which allows LangChain to communicate with OpenAI models.

**5.) langchain_core.prompts:**
- 5.1) *ChatPromptTemplate:*
    - Purpose: Create structured chat prompts containing different message types.
    - Example:
        ```
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful mathematics assistant."),
            ("human", "{input}")
        ])
        ```
- 5.2) *SystemMessagePromptTemplate:*
    - Purpose: Create reusable system message templates.
    - Example:
        ```
        system_prompt = SystemMessagePromptTemplate.from_template(
            "You are an expert {domain} assistant."
        )
        ```
- 5.3) *PromptTemplate:*
    - Purpose: Create reusable text-based prompts with dynamic variables.
    - Example:
        ```
        prompt = PromptTemplate.from_template(
            "Explain {topic} in simple language."
        )
        print(prompt.format(topic="Python decorators"))
        ```
    - Output: `Explain Python decorators in simple language.`
- 5.4) *MessagesPlaceholder:*
    - Purpose: Insert a list of messages into a chat prompt dynamically.
    - Example:
        ```
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful assistant."),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])
        ```

In [4]:
# Currently, we are having the data in the form of a dictionary inside our function. 
# In a real-world scenario, this data might come from a database or an API.

# Tool: A tool is simply a Python function that an AI Model or LLM can call to perform a specific task. 
# In this case, we are creating a tool that retrieves the schedule of a course based on its name.

@tool
def get_course_schedule(course_name: str) -> str:
    """
    Get the schedule for a given course.
    """
    schedule = {
        "python" : "Monday and Wednesday, 10:00 AM - 12:00 PM",
        "sql" : "Tuesday and Thursday, 2:00 PM - 4:00 PM",
        "data science" : "Friday, 1:00 PM - 3:00 PM"
    }

    key = course_name.lower()
    return schedule.get(key, "Course not found.") # if the specified key exists, it returns the corresponding value, otherwise it returns the string "Course not found."


- **LangChain's** *@tool* decorator requires a description for your tool.
- If you do not use docstring for a function/tool, then you will get error.
- See the example below:

In [5]:
@tool
def calculate_discounted_price(original_price: float, discounted_percentage: float) -> float:
    discount_amount = (discounted_percentage / 100) * original_price
    discounted_price = original_price - discount_amount
    return round(discounted_price, 2)  # Round to 2 decimal places

ValueError: Function must have a docstring if description not provided.

In [6]:
@tool
def calculate_discounted_price(original_price: float, discounted_percentage: float) -> float:
    """
    Calculate the discounted price given the original price and the discount percentage.
    """
    discount_amount = (discounted_percentage / 100) * original_price
    discounted_price = original_price - discount_amount
    return round(discounted_price, 2)  # Round to 2 decimal places

- If you do not want to use docstrings, you can also provide a description for the tool using the 
`description` parameter in the `@tool` decorator like this:

In [7]:
@tool(description="It is a real life tool which will help me out for next step recommendation for my learning path.")
def recommend_next_topic(topic: str) -> str:
    topic = topic.lower()
    
    if "python" in topic:
        return "next step for python is to learn web development using Django or Flask."
    elif "data science" in topic:
        return "next step for data science is to learn machine learning algorithms and their applications."
    elif "machine learning" in topic:
        return "next step for machine learning is to learn deep learning and neural networks."
    else:
        return "Sorry, I don't have a recommendation for that topic."

##### create_tool_calling_agent: 
- Read the official langchain docs.
- This langchain method is used to create an agent that uses tools.
- Requires llm, tools, prompts.
- Syntax: 
```
create_tool_calling_agent(
  llm: BaseLanguageModel,
  tools: Sequence[BaseTool],
  prompt: ChatPromptTemplate,
  *,
  message_formatter: MessageFormatter = format_to_tool_messages
) -> Runnable
```

In [8]:
tools = [get_course_schedule, calculate_discounted_price, recommend_next_topic]

In [10]:
# got it from EURI AI documentation on pypi to use EURIAPI key and avoid using OPENAI API key.
from euriai.langchain import EuriaiChatModel 

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True)) 

euri_api_key = os.environ["EURI_API_KEY"]


In [17]:
# we have already created 'tools', now need to have 'llm' and 'prompts/instructions'

# giving llm access, this comes from EURI API
llm = EuriaiChatModel(
    api_key=euri_api_key,
    model="gpt-4o-mini",
    temperature=0
)

In [18]:
from langchain_core.prompts import ChatPromptTemplate

# taken from langchain offcial docs
prompt = ChatPromptTemplate.from_messages([
    (
        "system", # Setting System Prompt
        """
        You are a helpful teaching assistant that helps students
        with learning paths and course-related queries.

        Use the available tools whenever they are relevant.

        Rules:
        1. If the question is about schedules, batches, or timings,
           ALWAYS use get_course_schedule.

        2. If the question is about prices, discounts, GST, or splits,
           ALWAYS use calculate_discounted_price.

        3. If the question is about what to learn next,
           ALWAYS use recommend_next_topic.

        4. If the user asks something outside these topics, respond:
           "Sorry, I can only help with course schedules,
           discounted prices, and learning path recommendations."

        5. Never invent course schedules or prices.
           Always use the appropriate tool to retrieve this information.

        6. After receiving the tool result, summarize it
           in a human-readable way.
        """
    ),
    ("user", "{input}"), # represents the user's current message; sometimes 'human' is also used instead of 'user'
    ("placeholder", "{agent_scratchpad}") # temporary working memory used by an AI agent to store intermediate thoughts, plans, tool calls, and observations while solving a single task
])

**agent_scratchpad:**
- Just like we take notes, agents internally take notes, like agents usually stores intermediate thoughts, tool calls, and tool outputs during a single task execution and then decides the next task and move on.
- This is where *agent_scratchpad* is used for.
- It is temporary memory.

Tools, LLMS and prompts are ready, now we need to integrate them together using **create_tool_calling_agent**

In [19]:
# Excited to create our first agent
agent = create_tool_calling_agent(
    llm = llm,
    tools = tools,
    prompt = prompt
)

Q.) Alright, we have created an agent but how to execute it ?

Answer: By using an **AgentExecutor** provided by Langchain.

In [20]:
# Library for executor is already imported in the starting
# AgentExecutor is the legacy, currently Langchain uses create_agent()

executor = AgentExecutor(agent=agent, tools=tools, verbose=True) 

# verbose = True helps in debugging the AI Agent by displaying execution logs.

In [21]:
# checking independently
result = get_course_schedule.invoke(
    {"course_name": "Data science"}
)

print(result)

Friday, 1:00 PM - 3:00 PM


In [22]:
demo_query = "I am learning Python. What should I learn next?"
response = executor.invoke({"input": demo_query })

# Agents will give incorrect answer for our query because although we have imported OPENAI Library with Langchain, but we are using EURI AI 
# integration with Langchain instead of OPEN AI to avoid generating OPEN AI API Key. Why ? Because we already have EURI API KEY so i thought of 
# using that only rather than generating a new API key of OPEN AI. But the problem is that EURI is not properly integrated with Langchain unlike 
# OPENAI,hence we are getting incorrect answer from the AI agent, therefore to get correct answer we need to make EURI compatible with langchain.



> Entering new AgentExecutor chain...
Let me recommend the next topic for you to learn in your Python journey. Please hold on for a moment. 

Using the recommendation tool... 

Based on your current learning path in Python, the next topic you should consider learning is "Object-Oriented Programming (OOP) in Python." This will help you understand how to create classes and objects, manage data, and implement inheritance, encapsulation, and polymorphism in your projects. Happy learning!

> Finished chain.


In [24]:
demo_query = "tell me the exact schedule for Data science course."
response = executor.invoke({"input": demo_query})



> Entering new AgentExecutor chain...
I'll check the schedule for the Data Science course for you. One moment please. 

[Using get_course_schedule]

> Finished chain.


## Why Is Our LangChain Agent Not Calling Tools Correctly?

### 1. Background

LangChain supports integrations with different LLM providers, including OpenAI and EURI AI.

In this project, we are using **EURI AI instead of OpenAI** because we already have an EURI API key and want to avoid generating a separate OpenAI API key.

We initialized the model using EURI's LangChain integration:

```python
from euriai.langchain import EuriaiChatModel

llm = EuriaiChatModel(
    api_key=euri_api_key,
    model="gpt-4.1-nano",
    temperature=0
)
```

### 2. Root Cause

Our agent uses LangChain's tool-calling mechanism to select and execute Python tools.

However, the EURI LangChain wrapper has implementation issues that prevent this mechanism from working correctly.

During debugging, we identified the following problems:

**Problem 1: Tool definitions are not retained**

- LangChain's `bind_tools()` converts our Python tools into function definitions.
- EURI's `bind_functions()` attempts to store these definitions in an internal attribute called `_bound_functions`.
- However, `_bound_functions` is not declared as a Pydantic field or private attribute.
- The model uses `extra="ignore"`, so Pydantic discards this additional attribute.
- As a result, the bound model does not retain the tool definitions.

**Problem 2: Model configuration parameter mismatch**

- We initially passed `model_name="gpt-4o-mini"`.
- However, EURI's model declares `model` as its configuration field, not `model_name`.
- Because additional fields are ignored, our requested model name was discarded.
- The model therefore used its default model, `gpt-4.1-nano`.

### 3. Impact on the Agent

The expected tool-calling process is:

1. The user submits a query.
2. The LLM identifies the appropriate tool.
3. The LLM returns a structured tool call with the tool name and arguments.
4. LangChain executes the selected Python tool.
5. The tool returns its result.
6. The LLM generates a human-readable response using that result.

In our implementation, the model returns ordinary text instead of a structured tool call.

For example:

```python
response = llm_with_tools.invoke(
    "Find the schedule for Data science."
)

print(response.tool_calls)

# Output: []
```

An empty `tool_calls` list means the model did not return a structured tool call. Consequently, LangChain's agent cannot execute the intended Python tool.

### 4. Conclusion

The issue is not with our Python tool definitions or their independent execution. We verified that the tools work when invoked directly.

The problem lies in the EURI LangChain integration and its tool-binding implementation.

To resolve this, we need either:

- A corrected EURI LangChain integration that properly retains tool definitions, sends them to the API, and converts tool-call responses into LangChain's expected format; or
- An alternative implementation that handles tool selection and execution manually.

**Key Learning:** Having a `bind_tools()` method does not necessarily mean that an LLM integration fully supports LangChain's native tool-calling workflow. We must verify that tool definitions reach the API and that structured tool calls are returned in the expected format.